# 01 · Build the dataset (image → JSON manifest)
This notebook demonstrates the **individually-exposed dataset utilities** in `gemma_ft_json.data.build_dataset` and assembles a train/val *manifest*.

Pipeline: `iter_image_paths → pair_image_json → validate_record → target_to_string → build_manifest → split_manifest`.

**Why a manifest?** It decouples slow disk scanning/validation from training. Each line is one record `{image_path, target}` where `target` is a *canonical* JSON string (sorted keys, no spaces) so the model learns one stable serialisation.

In [ ]:
# --- Bootstrap: make the package importable without installing, and stay OFFLINE.
import os, sys
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)


In [ ]:
# --- (Optional) create a tiny SYNTHETIC dataset so every notebook runs out-of-the-box.
# Replace these paths in configs/default.yaml with your REAL data to train for real.
import json, numpy as np
from PIL import Image
DEMO_IMAGES = ROOT / "data" / "demo" / "images"
DEMO_JSON   = ROOT / "data" / "demo" / "json"
DEMO_IMAGES.mkdir(parents=True, exist_ok=True); DEMO_JSON.mkdir(parents=True, exist_ok=True)
if not any(DEMO_IMAGES.glob("*.png")):
    rng = np.random.default_rng(0)
    for i in range(8):
        Image.fromarray((rng.random((80, 130, 3)) * 255).astype("uint8")).save(DEMO_IMAGES / f"sample_{i:03d}.png")
        json.dump({"rows": [{"item": f"row{i}", "qty": i, "price": round(1.5 * i, 2)}]},
                  open(DEMO_JSON / f"sample_{i:03d}.json", "w"))
    print("created 8 synthetic image/JSON pairs")
print("demo images:", DEMO_IMAGES)


In [ ]:
from gemma_ft_json.config import load_config
from gemma_ft_json.data.build_dataset import (
    iter_image_paths, pair_image_json, validate_record, target_to_string,
    build_manifest, split_manifest,
)

cfg = load_config(ROOT / 'configs' / 'default.yaml')
# Point the config at the demo data for this walkthrough (edit YAML for real runs).
cfg.paths.images_dir = str(DEMO_IMAGES)
cfg.paths.json_dir   = str(DEMO_JSON)
cfg.paths.manifest_dir = str(ROOT / 'data' / 'demo' / 'manifests')
print('images_dir =', cfg.paths.images_dir)

### Step 1 — discover images and pair each with its JSON ground truth

In [ ]:
paths = list(iter_image_paths(cfg.paths.images_dir))
print(f'found {len(paths)} images')
for p in paths[:3]:
    j = pair_image_json(p, cfg.paths.json_dir)
    print(p.name, '->', j.name if j else 'NO JSON')

### Step 2 — validate + canonicalise one target (what the model will learn to emit)

In [ ]:
import json
j = pair_image_json(paths[0], cfg.paths.json_dir)
obj = json.load(open(j))
print('valid record? ', validate_record(obj))
print('canonical target string:')
print(target_to_string(obj))

### Step 3 — build the full manifest, then split into train/val

In [ ]:
manifest_path, stats = build_manifest(
    cfg.paths.images_dir, cfg.paths.json_dir, cfg.paths.manifest_dir,
    require_valid_json=cfg.data.require_valid_json,
    use_pymupdf_boxes=cfg.data.use_pymupdf_boxes,
)
print('stats:', stats)
train_path, val_path = split_manifest(manifest_path, cfg.data.val_fraction, seed=cfg.project.seed)
print('train:', train_path)
print('val:  ', val_path)

In [ ]:
# Peek at the manifest lines.
for line in open(train_path).read().splitlines()[:2]:
    print(line[:200], '...')

✅ The manifests in `data/demo/manifests/` now feed notebook **02** (dataloader). For real training, set `paths.images_dir` / `paths.json_dir` in `configs/default.yaml` and re-run.